# Overcooked-AI — Entrenamiento **E3T-lite + PPO** (Colab)

Notebook autónomo que implementa el pipeline de `PLANEACION.md` (método principal:
**E3T-lite sobre PPO/SB3**, obs `lossless_grid` + CNN con padding, multi-layout, asiento
aleatorio, shaping annealado) y — lo importante para Colab — **checkpoints a Google Drive
con reanudación automática** por si se corta la sesión.

**Flujo:** instala → clona el repo → arma el *pool* de layouts (F0) → wrapper Gym +
compañero E3T-lite (F3) multi-layout (F4) → entrena con checkpoints → evalúa con el score
oficial → exporta el `StudentAgent` final.

> Para reanudar tras una desconexión: reconecta, ejecuta **Runtime → Run all**. La celda de
> entrenamiento detecta el último checkpoint en Drive y continúa desde ahí (`reset_num_timesteps=False`).


## 0. Configuración (edita aquí)

In [ ]:
# --- Repo ---
REPO_URL   = "https://github.com/Snah-s/deep_project.git"
REPO_DIR   = "deep_project"            # carpeta local tras clonar
# Si el repo es PRIVADO, pega un token de GitHub (Fine-grained, permiso contents:read).
# Deja "" si el repo es publico. El token NO se guarda en el notebook al compartirlo.
GITHUB_TOKEN = ""                      # ej: "github_pat_xxx"

# --- Drive / checkpoints ---
DRIVE_ROOT = "/content/drive/MyDrive/overcooked_e3t"

# --- Presupuesto de entrenamiento ---
import os
TOTAL_TIMESTEPS = 3_000_000            # objetivo (el plan usa ~10M; sube si tienes tiempo)
SUBPROC         = True                 # SubprocVecEnv: recoleccion en paralelo (mas rapido, mas CPU/RAM)
N_ENVS          = (os.cpu_count() or 8) if SUBPROC else 8   # 1 proceso por core cuando SUBPROC
# CLAVE de velocidad: las redes son diminutas -> torch reparte cada forward entre TODOS los
# cores y la coordinacion lo hace ~12x mas LENTO. 1 thread por proceso es lo optimo (medido).
TORCH_THREADS   = 1
HORIZON         = 400                  # largo de episodio de entrenamiento (eval usa 250 y 400)
SAVE_EVERY_STEPS = 100_000             # frecuencia de checkpoint a Drive (robusto a cortes)

# --- E3T-lite ---
E3T_EPSILON     = 0.5                  # prob. de accion aleatoria del companero (paper ~0.5)
FROZEN_REFRESH  = 200_000              # cada cuantos pasos se refresca la copia congelada del ego
SCRIPTED_PARTNER_FRAC = 0.15           # fraccion de episodios con companero scripted (greedy/stay/random)

# --- Shaping anti-atasco / progreso (idea adaptada de DoomBot) ---
# Reward auxiliar que empuja a NO quedarse pasivo/ciclando (debilidad medida: 0 sopas en solitario).
# Solo actua cuando NO hubo progreso real de la tarea y se annela junto al shaping (via shaping_coef).
PROGRESS_SHAPING = True
NOVELTY_BONUS    = 0.02   # bono por pisar una celda nueva en el episodio (exploracion)
STUCK_PEN        = 0.05   # penalizacion/paso cuando el estado (pos,orient,objeto) no cambia
STUCK_STEPS      = 8      # nº de pasos idénticos para considerarse "atascado"

# --- Watchdog anti-ciclo del StudentAgent entregable (fallback §6.6, idea de DoomBot) ---
WATCHDOG_STEPS   = 10     # si la obs no cambia por N pasos, inyecta accion de escape

# --- PPO ---
LR = 2.5e-4; N_STEPS = 400; N_EPOCHS = 10; CLIP = 0.2; GAMMA = 0.99; GAE = 0.95
ENT0, ENT1 = 0.10, 0.01               # entropia annealada 0.10 -> 0.01
SHAPING_END_FRAC = 0.6                 # el shaping se annela a 0 al ~60% del entrenamiento

# --- Pool de layouts ---
# "conocidos" (proxy escenarios 1-3) siempre en entrenamiento:
KNOWN_LAYOUTS   = ["cramped_room", "asymmetric_advantages", "coordination_ring"]
# held-out: NUNCA se entrenan; sirven de proxy de los escenarios secretos 4-6:
HELDOUT_LAYOUTS = ["counter_circuit", "forced_coordination"]
# Pool de ENTRENAMIENTO. Empezar con POCOS layouts: cada uno necesita muchos pasos
# (el paper E3T usa 10M en UNO solo). Repartir 3M entre 41 layouts -> no aprende.
# Recomendado: F3 = ["cramped_room"]; luego ampliar subiendo TOTAL_TIMESTEPS en proporcion.
# Deja [] para usar TODOS los layouts validos del paquete menos los held-out.
TRAIN_LAYOUTS   = list(KNOWN_LAYOUTS)
DIM_CAP         = 15                   # descarta layouts con lado > DIM_CAP (mantiene el padding chico)
PAD_MARGIN      = 2                    # margen extra de padding para layouts algo mas grandes

SEED = 0
print("Config cargada.")

## 1. Instalar dependencias

`overcooked-ai` requiere `numpy<2`. Si Colab pide reiniciar el runtime tras el downgrade de numpy, hazlo (**Runtime → Restart**) y vuelve a **Run all**.

In [ ]:
import subprocess, sys
def pip(*args):
    print(">> pip", *args)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("overcooked-ai")
pip("stable-baselines3>=2.0.0")
pip("gymnasium>=0.29")
pip("PyYAML>=6.0", "Pillow>=10.0", "imageio>=2.31")
pip("numpy<2")   # ultimo: fija numpy<2 por encima de lo que hayan traido las otras libs

import numpy as np
print("numpy", np.__version__)
assert np.__version__.startswith("1."), "Reinicia el runtime (Runtime > Restart) y vuelve a Run all."

## 2. Clonar el repo y preparar el `PYTHONPATH`

In [ ]:
import os, sys, subprocess

url = REPO_URL
if GITHUB_TOKEN:
    url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", url, REPO_DIR], check=True)
else:
    print("Repo ya clonado.")

# El codigo del proyecto vive en <repo>/overcooked con paquete importable `src`
PKG_DIR = os.path.abspath(os.path.join(REPO_DIR, "overcooked"))
os.chdir(PKG_DIR)
if PKG_DIR not in sys.path:
    sys.path.insert(0, PKG_DIR)
print("cwd:", os.getcwd())

# Sanity: importar utilidades del repo que vamos a reutilizar
from src.environment import build_mdp
from src.constants import action_index_to_overcooked_action, INDEX_TO_OVERCOOKED_ACTION
from policies.basic_policies import GreedyFullTaskPolicy, StayPolicy, RandomMotionPolicy
print("Imports del repo OK. Acciones:", {i: str(a) for i, a in INDEX_TO_OVERCOOKED_ACTION.items()})

## 3. Montar Google Drive (checkpoints)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CKPT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
EXPORT_DIR = os.path.join(DRIVE_ROOT, "export")
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)
print("Checkpoints ->", CKPT_DIR)

## 4. Armar el *pool* de layouts (F0)

Se recorren los layouts instalados con `overcooked-ai`, se valida que sean de 2 jugadores, que produzcan **26 canales** (recetas de cebolla / `old_dynamics`, canales consistentes) y que quepan en `DIM_CAP`. Los `HELDOUT_LAYOUTS` se reservan para evaluar generalización.

In [ ]:
import os, glob
import numpy as np
from overcooked_ai_py.mdp.overcooked_env import OvercookedEnv
import overcooked_ai_py

_MDP_CACHE = {}
def get_mdp(layout_name):
    if layout_name not in _MDP_CACHE:
        _MDP_CACHE[layout_name] = build_mdp({"layout_name": layout_name, "old_dynamics": True})
    return _MDP_CACHE[layout_name]

REF_CHANNELS = 26  # cramped_room / cebolla, old_dynamics

def layout_ok(name):
    try:
        mdp = get_mdp(name)
        if mdp.num_players != 2:
            return None
        env = OvercookedEnv.from_mdp(mdp, horizon=HORIZON, info_level=0); env.reset()
        arr = np.asarray(mdp.lossless_state_encoding(env.state, horizon=HORIZON)[0])
        d0, d1, c = arr.shape
        if c != REF_CHANNELS: return None
        if max(d0, d1) > DIM_CAP: return None
        return (d0, d1, c)
    except Exception:
        return None

# Enumerar los layouts del paquete
lay_dir = os.path.join(os.path.dirname(overcooked_ai_py.__file__), "data", "layouts")
all_names = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(lay_dir, "*.layout")))

valid = {}
for n in all_names:
    shp = layout_ok(n)
    if shp: valid[n] = shp

HELDOUT = [n for n in HELDOUT_LAYOUTS if n in valid]
if TRAIN_LAYOUTS:
    TRAIN_POOL = [n for n in TRAIN_LAYOUTS if n in valid and n not in HELDOUT]
    missing = [n for n in TRAIN_LAYOUTS if n not in valid]
    if missing: print("AVISO: layouts pedidos no validos (omitidos):", missing)
else:
    TRAIN_POOL = [n for n in valid if n not in HELDOUT]
assert TRAIN_POOL, "Pool de entrenamiento vacio: revisa TRAIN_LAYOUTS"

# Dimensiones de padding sobre TODO el universo (train + heldout) + margen
dims = np.array([valid[n][:2] for n in valid])
P0 = int(dims[:, 0].max()) + PAD_MARGIN
P1 = int(dims[:, 1].max()) + PAD_MARGIN
C  = REF_CHANNELS
OBS_SHAPE = (C, P0, P1)

print(f"Layouts validos: {len(valid)} | train: {len(TRAIN_POOL)} | held-out: {HELDOUT}")
print("TRAIN_POOL:", TRAIN_POOL)
print("Obs shape (C,P0,P1):", OBS_SHAPE)
assert len(TRAIN_POOL) >= 1

## 5. Codificador + padding compartido

La **misma** función se usa para la obs del agente, la del compañero E3T y el `StudentAgent` final: garantiza que lo que ve en entrenamiento es lo que verá en el torneo.

In [ ]:
def encode_padded(mdp, state, agent_index, horizon=HORIZON):
    """Lossless encoding del asiento -> (C, P0, P1) float32, con padding a tamano fijo."""
    arr = np.asarray(mdp.lossless_state_encoding(state, horizon=horizon)[agent_index], dtype=np.float32)
    d0, d1, c = arr.shape
    out = np.zeros((C, P0, P1), dtype=np.float32)
    dd0, dd1 = min(d0, P0), min(d1, P1)
    out[:c, :dd0, :dd1] = np.transpose(arr[:dd0, :dd1, :], (2, 0, 1))
    return out

## 6. Wrapper Gym single-agent + compañeros (E3T-lite)

PPO controla un asiento (aleatorio por episodio); el otro lo llena un compañero intercambiable. Para E3T el compañero es una **copia congelada del ego** con probabilidad `ε` de acción aleatoria. La recompensa es **sparse de equipo + shaping propio** (el coeficiente lo annela un callback).

In [ ]:
import gymnasium as gym
from gymnasium import spaces
from overcooked_ai_py.agents.agent import Agent

# Espacios a nivel de modulo (los usan el env y la copia congelada de cada worker)
OBS_SPACE = spaces.Box(0.0, 50.0, shape=OBS_SHAPE, dtype=np.float32)
ACT_SPACE = spaces.Discrete(6)

class WorkerFrozen:
    """Copia congelada del ego PROPIA de cada entorno. Compatible con SubprocVecEnv:
    el callback difunde el state_dict a cada worker (no hay objeto compartido)."""
    def __init__(self): self.policy = None; self.ready = False
    def load(self, state_dict):
        if self.policy is None:  # construccion perezosa (POLICY_KWARGS ya existe al entrenar)
            from stable_baselines3.common.policies import ActorCriticPolicy
            self.policy = ActorCriticPolicy(OBS_SPACE, ACT_SPACE, lambda _: 0.0, **POLICY_KWARGS)
            self.policy.to("cpu").eval()  # el companero infiere en CPU dentro del worker
        self.policy.load_state_dict(state_dict); self.ready = True

class FrozenEgoAgent(Agent):
    """Companero = copia congelada del ego. Codifica su propio asiento y consulta la red."""
    def __init__(self, frozen, rng): super().__init__(); self.frozen = frozen; self.rng = rng
    def action(self, state):
        if not self.frozen.ready:  # aun sin pesos -> movimiento aleatorio
            idx = int(self.rng.integers(0, 6))
            return action_index_to_overcooked_action(idx), {"policy_name": "frozen_ego_random"}
        obs = encode_padded(self.mdp, state, self.agent_index)
        a, _ = self.frozen.policy.predict(obs[None], deterministic=False)
        return action_index_to_overcooked_action(int(a[0])), {"policy_name": "frozen_ego"}

class EpsilonAgent(Agent):
    """Envuelve a otro agente y con prob epsilon devuelve accion uniforme (mezcla E3T)."""
    def __init__(self, base, epsilon, rng):
        super().__init__(); self.base = base; self.epsilon = float(epsilon); self.rng = rng
    def set_mdp(self, mdp): super().set_mdp(mdp); self.base.set_mdp(mdp)
    def set_agent_index(self, i): super().set_agent_index(i); self.base.set_agent_index(i)
    def reset(self):
        super().reset()
        if hasattr(self, "base"): self.base.reset()
    def action(self, state):
        if self.rng.random() < self.epsilon:
            return action_index_to_overcooked_action(int(self.rng.integers(0, 6))), {"policy_name": "eps_random"}
        return self.base.action(state)

def make_partner(frozen, rng):
    """Muestrea el companero de un episodio: E3T (por defecto) o scripted (regularizacion/robustez)."""
    if rng.random() < SCRIPTED_PARTNER_FRAC:
        pick = rng.integers(0, 3)
        if pick == 0: return GreedyFullTaskPolicy(seed=int(rng.integers(1e9)))
        if pick == 1: return StayPolicy()
        return RandomMotionPolicy(seed=int(rng.integers(1e9)))
    return EpsilonAgent(FrozenEgoAgent(frozen, rng), E3T_EPSILON, rng)

class OvercookedGymEnv(gym.Env):
    """Overcooked de 2 jugadores como entorno single-agent Gymnasium."""
    metadata = {"render_modes": []}
    def __init__(self, seed=0):
        super().__init__()
        self.frozen = WorkerFrozen()  # copia congelada propia de este worker
        self.shaping_coef = 1.0
        self.observation_space = OBS_SPACE
        self.action_space = ACT_SPACE
        self._rng = np.random.default_rng(seed)

    def set_shaping_coef(self, v): self.shaping_coef = float(v)
    def load_frozen(self, state_dict): self.frozen.load(state_dict)  # llamado por el callback

    def reset(self, *, seed=None, options=None):
        if seed is not None: self._rng = np.random.default_rng(seed)
        self.layout = self.TRAIN_POOL[int(self._rng.integers(0, len(self.TRAIN_POOL)))]
        self.mdp = get_mdp(self.layout)
        self.oc = OvercookedEnv.from_mdp(self.mdp, horizon=HORIZON, info_level=0)
        self.oc.reset()
        self.seat = int(self._rng.integers(0, 2))
        self.partner = make_partner(self.frozen, self._rng)
        # Orden como en el runner del repo: reset() ANTES de set_mdp (reset limpia self.mdp).
        self.partner.reset(); self.partner.set_mdp(self.mdp); self.partner.set_agent_index(1 - self.seat)
        self._visited = set(); self._last_sig = None; self._stuck = 0  # tracking anti-atasco
        obs = encode_padded(self.mdp, self.oc.state, self.seat)
        return obs, {"layout": self.layout, "seat": self.seat}

    def _aux_reward(self, next_state, sparse_team, shaped_own):
        """Reward de exploracion/anti-atasco (solo si no hubo progreso real de la tarea)."""
        if not PROGRESS_SHAPING or sparse_team != 0 or shaped_own != 0:
            self._stuck = 0
            return 0.0
        try:
            p = next_state.players[self.seat]
            sig = (p.position, p.orientation, None if p.held_object is None else p.held_object.name)
            aux = 0.0
            if p.position not in self._visited:
                self._visited.add(p.position); aux += NOVELTY_BONUS
            if sig == self._last_sig:
                self._stuck += 1
                if self._stuck >= STUCK_STEPS: aux -= STUCK_PEN
            else:
                self._stuck = 0
            self._last_sig = sig
            return aux
        except Exception:
            return 0.0

    def step(self, action):
        state = self.oc.state
        our = action_index_to_overcooked_action(int(action))
        p_act, _ = self.partner.action(state)
        joint = [None, None]; joint[self.seat] = our; joint[1 - self.seat] = p_act
        next_state, sparse_team, done, info = self.oc.step(tuple(joint))
        shaped_own = float(info["shaped_r_by_agent"][self.seat])
        aux = self._aux_reward(next_state, sparse_team, shaped_own)   # anti-atasco (DoomBot)
        reward = float(sparse_team) + self.shaping_coef * (shaped_own + aux)  # aux se annela con el shaping
        obs = encode_padded(self.mdp, next_state, self.seat)
        info["sparse_team"] = float(sparse_team)
        return obs, reward, bool(done), False, info

# Inyectar el pool como atributo de clase (evita capturarlo por closure)
OvercookedGymEnv.TRAIN_POOL = TRAIN_POOL
print("Wrapper Gym listo.")

## 7. Red CNN pequeña + PPO + callbacks

Extractor CNN de 3 conv 3×3 (`normalize_images=False`: la obs no es una imagen 0-255). Un callback difunde la copia congelada del ego a los workers y annela shaping y entropía **una vez por rollout** (no por paso: con `SubprocVecEnv` cada llamada cruza procesos). Con `SUBPROC=True` la recolección corre en `N_ENVS` procesos en paralelo → más rápido a cambio de más CPU/RAM; el GPU sigue holgado (el cuello es la CPU del entorno).

In [ ]:
import torch, torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
torch.backends.cudnn.benchmark = True   # kernels mas rapidos para tamano de obs fijo
torch.set_num_threads(TORCH_THREADS)    # ~12x en la recoleccion con redes chicas (ver config)

class SmallCNN(BaseFeaturesExtractor):
    def __init__(self, obs_space, features_dim=64):
        super().__init__(obs_space, features_dim)
        c = obs_space.shape[0]
        self.cnn = nn.Sequential(
            nn.Conv2d(c, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.Flatten())
        with torch.no_grad():
            n = self.cnn(torch.zeros(1, *obs_space.shape)).shape[1]
        self.linear = nn.Sequential(nn.Linear(n, features_dim), nn.ReLU())
    def forward(self, x): return self.linear(self.cnn(x))

POLICY_KWARGS = dict(
    features_extractor_class=SmallCNN,
    features_extractor_kwargs=dict(features_dim=64),
    net_arch=[64, 64],
    normalize_images=False,
)

class E3TCallback(BaseCallback):
    """Difunde la copia congelada del ego a los workers y annela shaping + entropia.
    Trabaja UNA vez por rollout (no por paso) para no saturar el IPC con SubprocVecEnv."""
    def __init__(self, total):
        super().__init__(); self.total = total; self._last_refresh = -1
    def _on_rollout_start(self):
        t = self.num_timesteps
        if self._last_refresh < 0 or (t - self._last_refresh) >= FROZEN_REFRESH:
            sd = {k: v.detach().cpu() for k, v in self.model.policy.state_dict().items()}
            self.training_env.env_method("load_frozen", sd)  # broadcast a cada worker
            self._last_refresh = t
        frac = min(1.0, t / self.total)
        self.training_env.env_method("set_shaping_coef", max(0.0, 1.0 - frac / SHAPING_END_FRAC))
        self.model.ent_coef = ENT0 + (ENT1 - ENT0) * frac
    def _on_step(self): return True

def _make_env_fn(rank):
    def _f():
        if SUBPROC:
            import torch as _t; _t.set_num_threads(1)  # evita oversubscription entre workers
        return Monitor(OvercookedGymEnv(seed=SEED * 1000 + rank))
    return _f

def make_venv():
    fns = [_make_env_fn(i) for i in range(N_ENVS)]
    if SUBPROC and N_ENVS > 1:
        return SubprocVecEnv(fns, start_method="fork")  # fork: comparte las clases del notebook
    return DummyVecEnv(fns)

print(f"VecEnv: {'SubprocVecEnv' if (SUBPROC and N_ENVS > 1) else 'DummyVecEnv'} | N_ENVS={N_ENVS}")

## 8. Entrenar con checkpoints a Drive (reanudable)

Si hay un checkpoint en Drive, **reanuda**; si no, empieza de cero. Reejecutar esta celda tras una desconexión continúa desde el último `.zip` guardado. El `CheckpointCallback` vuelca a Drive cada `SAVE_EVERY_STEPS`.

In [ ]:
import re
venv = make_venv()                                   # crear (y forkear los workers) ANTES de iniciar CUDA
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

def latest_ckpt(d):
    zips = glob.glob(os.path.join(d, "e3t_*_steps.zip"))
    if not zips: return None
    def steps(p):
        m = re.search(r"_(\d+)_steps\.zip$", p); return int(m.group(1)) if m else -1
    return max(zips, key=steps)

ckpt = latest_ckpt(CKPT_DIR)
if ckpt:
    print("Reanudando desde:", ckpt)
    model = PPO.load(ckpt, env=venv, device=device,
                     custom_objects={"policy_kwargs": POLICY_KWARGS})
else:
    print("Entrenando desde cero.")
    model = PPO("MlpPolicy", venv, policy_kwargs=POLICY_KWARGS,
                learning_rate=LR, n_steps=N_STEPS, batch_size=N_ENVS * N_STEPS // 4,
                n_epochs=N_EPOCHS, gamma=GAMMA, gae_lambda=GAE, clip_range=CLIP,
                ent_coef=ENT0, vf_coef=0.5, max_grad_norm=0.5, seed=SEED,
                device=device, verbose=1, tensorboard_log=os.path.join(DRIVE_ROOT, "tb"))

# La copia congelada de cada worker se siembra en el primer rollout (E3TCallback la difunde).
checkpoint_cb = CheckpointCallback(
    save_freq=max(SAVE_EVERY_STEPS // N_ENVS, 1),
    save_path=CKPT_DIR, name_prefix="e3t")
e3t_cb = E3TCallback(TOTAL_TIMESTEPS)

remaining = max(TOTAL_TIMESTEPS - model.num_timesteps, 0)
print(f"Pasos hechos: {model.num_timesteps:,} | restantes: {remaining:,}")
if remaining > 0:
    model.learn(total_timesteps=remaining, callback=[checkpoint_cb, e3t_cb],
                reset_num_timesteps=False, progress_bar=True)
    model.save(os.path.join(EXPORT_DIR, "e3t_final"))
    print("Entrenamiento terminado. Modelo final en:", EXPORT_DIR)
else:
    print("Ya se alcanzo el presupuesto de pasos.")

## 9. Evaluación con el score oficial

Evalúa en layouts **conocidos** y **held-out** (proxy de escenarios secretos), ambos asientos, vs compañeros held-out (greedy / stay / random). Reporta sopas y el **score oficial**:

`Score = 10000·sopas + 10·(H − t_última) + (H − t_primera) − min(100·timeouts, 5000)`

In [ ]:
import pandas as pd

def eval_episode(model, mdp, partner, seat, horizon, seed):
    env = OvercookedEnv.from_mdp(mdp, horizon=horizon, info_level=0); env.reset()
    partner.reset(); partner.set_mdp(mdp); partner.set_agent_index(1 - seat)
    soups, t_first, t_last = 0, None, None
    for t in range(horizon):
        obs = encode_padded(mdp, env.state, seat, horizon=horizon)
        a, _ = model.predict(obs[None], deterministic=True)
        our = action_index_to_overcooked_action(int(a[0]))
        p_act, _ = partner.action(env.state)
        joint = [None, None]; joint[seat] = our; joint[1 - seat] = p_act
        _, sparse_team, done, _ = env.step(tuple(joint))
        if sparse_team > 0:
            n = int(round(sparse_team / 20.0)); soups += n
            if t_first is None: t_first = t
            t_last = t
        if done: break
    if soups == 0: return 0.0, 0
    score = 10000 * soups + 10 * (horizon - t_last) + (horizon - t_first)
    return float(score), soups

def make_eval_partner(kind, seed):
    if kind == "greedy": return GreedyFullTaskPolicy(seed=seed)
    if kind == "stay":   return StayPolicy()
    return RandomMotionPolicy(seed=seed)

rows = []
for split, layouts in [("known", KNOWN_LAYOUTS), ("held-out", HELDOUT)]:
    for lay in layouts:
        if lay not in valid: continue
        mdp = get_mdp(lay)
        for pk in ["greedy", "stay", "random"]:
            for H in [250, 400]:
                sc, sp = [], []
                for seat in [0, 1]:
                    for sd in [67, 68, 69]:
                        s, k = eval_episode(model, mdp, make_eval_partner(pk, sd), seat, H, sd)
                        sc.append(s); sp.append(k)
                rows.append(dict(split=split, layout=lay, partner=pk, horizon=H,
                                 soups=np.mean(sp), score=np.mean(sc)))

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print("\nResumen por split/partner (score medio):")
print(df.groupby(["split", "partner"])["score"].mean().round(1).to_string())

## 10. Exportar el `StudentAgent` final

Escribe `policies/student_agent_e3t.py` + copia los pesos. El agente carga el modelo en `__init__`, **calienta** el forward en `reset()`, y `act()` padea la obs y devuelve un `int`. Incluye la **red de seguridad en capas**: cualquier excepción → `stay` (nunca rompe el runner). El fallback heurístico completo (parseo del grid, §6.6) se deja marcado como TODO para F5.

In [ ]:
import shutil

# Copiar pesos junto al .py (para que el entregable sea autocontenido)
model_zip = os.path.join(EXPORT_DIR, "e3t_final.zip")
if not os.path.exists(model_zip):
    model.save(os.path.join(EXPORT_DIR, "e3t_final"))
dst_zip = os.path.join(PKG_DIR, "policies", "e3t_model.zip")
shutil.copy(model_zip, dst_zip)

meta = dict(obs_shape=list(OBS_SHAPE), P0=P0, P1=P1, C=C, horizon=HORIZON,
            channels=REF_CHANNELS, train_pool=TRAIN_POOL, heldout=HELDOUT,
            watchdog_steps=WATCHDOG_STEPS)
import json as _json
with open(os.path.join(PKG_DIR, "policies", "e3t_meta.json"), "w") as f:
    _json.dump(meta, f, indent=2)

STUDENT_SRC = r"""
# StudentAgent E3T-lite: politica PPO (SB3) + fallback en capas.
from __future__ import annotations
import json, os
from collections import deque
import numpy as np

_HERE = os.path.dirname(os.path.abspath(__file__))

def _build_smallcnn():
    # La MISMA CNN del entrenamiento. Hay que reconstruirla: SB3 no recupera la clase
    # custom por cloudpickle entre versiones de Python (Colab 3.12 -> eval 3.10).
    import torch
    import torch.nn as nn
    from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
    class SmallCNN(BaseFeaturesExtractor):
        def __init__(self, obs_space, features_dim=64):
            super().__init__(obs_space, features_dim)
            c = obs_space.shape[0]
            self.cnn = nn.Sequential(
                nn.Conv2d(c, 32, 3, padding=1), nn.ReLU(),
                nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
                nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
                nn.Flatten())
            with torch.no_grad():
                n = self.cnn(torch.zeros(1, *obs_space.shape)).shape[1]
            self.linear = nn.Sequential(nn.Linear(n, features_dim), nn.ReLU())
        def forward(self, x):
            return self.linear(self.cnn(x))
    return SmallCNN

class StudentAgent:
    def __init__(self, config=None):
        self.config = config or {}
        with open(os.path.join(_HERE, "e3t_meta.json")) as f:
            self.meta = json.load(f)
        self.C, self.P0, self.P1 = self.meta["C"], self.meta["P0"], self.meta["P1"]
        self.watchdog_steps = int(self.meta.get("watchdog_steps", 10))
        self._hist = deque(maxlen=self.watchdog_steps)
        self._break_i = 0
        self.model = None
        try:
            from stable_baselines3 import PPO
            SmallCNN = _build_smallcnn()
            policy_kwargs = dict(features_extractor_class=SmallCNN,
                                 features_extractor_kwargs=dict(features_dim=64),
                                 net_arch=[64, 64], normalize_images=False)
            # custom_objects: reconstruye la politica sin depender del cloudpickle guardado.
            custom_objects = {"policy_kwargs": policy_kwargs, "lr_schedule": lambda _: 0.0,
                              "clip_range": lambda _: 0.2, "clip_range_vf": None}
            self.model = PPO.load(os.path.join(_HERE, "e3t_model.zip"),
                                  device="cpu", custom_objects=custom_objects)
            self.reset()  # calentar el forward
        except Exception as exc:
            print("StudentAgent: no se pudo cargar el modelo, uso fallback:", repr(exc))

    def reset(self):
        self._hist.clear(); self._break_i = 0
        if self.model is not None:
            try:
                self.model.predict(np.zeros((1, self.C, self.P0, self.P1), np.float32), deterministic=True)
            except Exception:
                pass

    def _pad(self, arr):
        arr = np.asarray(arr, dtype=np.float32)
        d0, d1, c = arr.shape
        out = np.zeros((self.C, self.P0, self.P1), np.float32)
        dd0, dd1 = min(d0, self.P0), min(d1, self.P1)
        out[:min(c, self.C), :dd0, :dd1] = np.transpose(arr[:dd0, :dd1, :min(c, self.C)], (2, 0, 1))
        return out

    def act(self, obs):
        try:
            arr = obs["obs"] if isinstance(obs, dict) else obs
            x = self._pad(arr)
            a, _ = self.model.predict(x[None], deterministic=True)
            action = int(a[0])
            # Watchdog anti-ciclo (idea de DoomBot): si la obs no cambia por N pasos, romper el bucle.
            self._hist.append(hash(x.tobytes()))
            if len(self._hist) == self._hist.maxlen and len(set(self._hist)) == 1:
                self._break_i = (self._break_i + 1) % 5
                return [5, 0, 1, 2, 3][self._break_i]  # interact, luego moverse
            return action
        except Exception:
            # Capa 1: nunca romper el runner. TODO F5: heuristica reactiva parseando el grid (§6.6).
            return 4  # stay
"""
with open(os.path.join(PKG_DIR, "policies", "student_agent_e3t.py"), "w") as f:
    f.write(STUDENT_SRC)

print("Exportado:")
print(" -", dst_zip)
print(" -", os.path.join(PKG_DIR, "policies", "e3t_meta.json"))
print(" -", os.path.join(PKG_DIR, "policies", "student_agent_e3t.py"))
print("\nPara evaluarlo con el runner: en configs/evaluate.yaml -> observation.type: lossless_grid,")
print("policies.agent_X.path: policies/student_agent_e3t.py, class_name: StudentAgent.")

## Notas

- **Reanudar tras corte de Colab:** reconecta y **Run all**. La celda 8 detecta el último
  checkpoint en Drive y continúa (`reset_num_timesteps=False`).
- **Subir el presupuesto:** el plan apunta a ~10M pasos/config; con GPU cabe holgado. Ajusta
  `TOTAL_TIMESTEPS` y reejecuta la celda 8 — reanuda desde donde iba.
- **Fidelidad a E3T:** compañero = copia congelada del ego + ε de acción aleatoria
  (`E3T_EPSILON≈0.5`), refrescada cada `FROZEN_REFRESH` pasos; entropía y shaping annealados.
- **Generalización:** entrena multi-layout con asiento aleatorio; evalúa en `HELDOUT_LAYOUTS`
  (nunca vistos) como proxy de los escenarios secretos 4-6.
- **Anti-atasco (idea de DoomBot):** en entrenamiento, un reward auxiliar (bono por explorar
  celdas nuevas + penalización por quedarse en el mismo estado) empuja al agente a no quedarse
  pasivo; se annela con el shaping. En el `StudentAgent`, un **watchdog** detecta si la obs no
  cambia por `WATCHDOG_STEPS` pasos e inyecta acciones de escape → protege contra el peor caso
  (score 0 por ciclar). Ambos configurables arriba.
- **Pendiente (F5):** heurística de fallback que parsea el grid; barridos de ε y FCP-lite
  reciclando checkpoints (los `.zip` en Drive ya sirven de población).

### Velocidad / uso de recursos
- El GPU casi no se usa (redes ~10⁵ params, ~800 MB de VRAM = solo overhead de CUDA). **El cuello
  es la CPU**: la inferencia del compañero y el forward de recolección.
- **`TORCH_THREADS=1`** es el mayor acelerador: con redes diminutas, repartir cada forward entre
  todos los cores lo hace ~12× más lento (medido: 7.5 ms con 16 threads vs 0.62 ms con 1).
- **`SUBPROC=True`** (`SubprocVecEnv`, `N_ENVS=cores`) paraleliza la recolección en varios procesos,
  cada uno a 1 thread → escala con los cores del runtime (más CPU/RAM a cambio). Con `SUBPROC=False`
  vuelve a `DummyVecEnv` (un proceso, útil para depurar).
- Si el runtime **no tiene GPU**, sube `TORCH_THREADS` a 2-4 solo si el *update* domina; normalmente
  conviene dejarlo en 1 porque manda la recolección.
